# A04 — diabetes: dữ liệu và so sánh đã lưu

Các framework dùng hai môi trường WSL riêng. Mã huấn luyện ở `training/diabetes_keras.ipynb` và `training/diabetes_torch.ipynb`. Notebook này đọc kết quả gốc; Run All không train/TEST. Toàn bộ notebook huấn luyện nguyên byte và output cũ nằm trong provenance.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'torch')
_migration


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Diabetes: dùng đặc trưng sức khỏe của bệnh nhân/người trả lời khảo sát để dự đoán `Diabetes_binary`. A04 dùng dataset BRFSS 2021 unbalanced đã chuẩn bị trong `datasets/diabetes/a04`, tập trung vào deep learning trên dữ liệu tabular và so sánh cẩn thận với A03.


Output cho thấy điều gì?

Bảng xác nhận notebook giữ đúng continuity với A03: cùng target `Diabetes_binary`, cùng domain patient health indicators, nhưng chuyển sang dataset A04 BRFSS 2021 lớn hơn và mất cân bằng hơn.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để tránh hard-code đường dẫn cá nhân.


Output cho thấy điều gì?

Environment có đủ TensorFlow và PyTorch để chạy cùng một pipeline dữ liệu. Nếu device là CPU thì notebook vẫn hợp lệ vì mô hình tabular nhỏ và batch size được chọn để chạy được trên laptop/Colab.


# 3. Load Dataset

Cell này làm gì?

Đọc raw CSV A04 từ `datasets/diabetes/a04` và giữ nguyên file raw. Mọi cleaning/preprocessing chỉ thực hiện trong DataFrame làm việc.


Output cho thấy điều gì?

Dataset được đọc từ path tương đối trong project. Shape là số dòng raw và số cột trước khi xử lý duplicate, còn `head()` xác nhận các feature đều là health indicators dạng numeric-coded tabular.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra schema, missing values, duplicates, target distribution, descriptive statistics và các vấn đề dữ liệu rõ ràng của Diabetes.


Output cho thấy điều gì?

Dataset A04 không có missing values nhưng có duplicate rows và class imbalance rõ: class 0 chiếm đa số. BMI không có giá trị 0 vô lý; `MentHlth` và `PhysHlth` có 0 hợp lệ vì đó là số ngày sức khỏe kém trong tháng.


# 5. Data Preparation

Cell này làm gì?

Loại exact duplicates trước split để tránh duplicate leakage, chia train/validation/test theo stratify, rồi fit imputer/scaler chỉ trên training data.


Output cho thấy điều gì?

Split giữ tỷ lệ class gần như giống nhau nhờ stratify. Preprocessor đã `fit` trên train rồi mới transform validation/test, nên không đưa thông tin từ validation/test vào bước scaling.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của tensor đầu vào deep learning và đánh giá representation có phù hợp với CNN không.


Output cho thấy điều gì?

`X_train` có dạng `(samples, features)`: mỗi dòng là một người trả lời khảo sát, mỗi cột là một biến sức khỏe. Vì các cột không tạo thành ảnh hay chuỗi có thứ tự lân cận tự nhiên, MLP là baseline đúng phương pháp hơn CNN.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa các operation nhỏ: Dense/Linear, ReLU, logits, sigmoid, Conv1D và pooling trên toy tensors.


Output cho thấy điều gì?

Dense biến vector feature thành logits; sigmoid chuyển logits sang xác suất cho binary classification. Conv1D/pooling chỉ được minh họa trên toy sequence có thứ tự, không áp đặt lên BRFSS tabular features.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, sigmoid, binary cross entropy, forward composition và Conv1D toy.


Output cho thấy điều gì?

NumPy forward pass tạo xác suất từ tabular features giống ý tưởng DNN trong A03. Conv1D thủ công chỉ minh họa phép trượt kernel trên một vector toy có thứ tự, không dùng như mô hình chính cho dataset BRFSS.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric, threshold selection và baseline Keras MLP cho Diabetes.


Output cho thấy điều gì?

Keras baseline nhận input 21 feature và trả về một xác suất `Diabetes_binary=1`. Số parameter nhỏ so với số sample, phù hợp với dữ liệu tabular và giảm nguy cơ overfitting quá mạnh.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, class weight, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


Output cho thấy điều gì?

Training log cho thấy loss/metric theo từng epoch trên train và validation. EarlyStopping/ModelCheckpoint chọn epoch có validation AUC tốt nhất, còn test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và AUC của baseline Keras trên train/validation, đồng thời lưu figure.


Output cho thấy điều gì?

Curves giúp kiểm tra mô hình học ổn định hay overfit. Khoảng cách train/validation là tín hiệu quan trọng hơn một epoch đơn lẻ.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng threshold chọn từ validation F1, gồm Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix và classification report.


Output cho thấy điều gì?

Validation metrics dùng nhiều góc nhìn, không chỉ accuracy. Với bài toán diabetes, false negatives đáng chú ý vì là các ca có diabetes/risk nhưng model dự đoán âm tính; tuy vậy đây vẫn là phân tích trên survey dataset, không phải chẩn đoán y khoa.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước khi train, chạy một batch qua model để kiểm tra input/logits shape.


Output cho thấy điều gì?

Batch input có dạng `(batch, 21)` và logits có dạng `(batch, 1)`, đúng với binary classification. PyTorch model dùng architecture tương đương Keras baseline để so sánh framework công bằng hơn.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline với loop thủ công, class imbalance dùng `pos_weight`, và lưu checkpoint tốt nhất theo validation ROC-AUC.


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ các bước `zero_grad`, forward, loss, backward và optimizer step. Checkpoint tốt nhất được chọn bằng validation ROC-AUC, không dùng test set.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng protocol như Keras.


Output cho thấy điều gì?

PyTorch được đánh giá trên cùng validation set và cùng cách chọn threshold theo F1. Vì framework, initialization và optimizer implementation có khác biệt nhỏ, một run không đủ để kết luận framework nào tốt hơn về bản chất.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng cùng metric validation.


Output cho thấy điều gì?

Bảng chỉ so sánh hai implementation baseline trong điều kiện hiện tại. Chênh lệch nhỏ có thể đến từ initialization, training loop và numerical behavior; không nên kết luận framework nào intrinsically better từ một run.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture MLP phù hợp dữ liệu tabular: M1 Basic MLP, M2 BatchNorm MLP, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được ép vào dữ liệu BRFSS vì feature order không có local structure tự nhiên.


Output cho thấy điều gì?

Các experiment giữ cùng split, preprocessing, seed và training budget. CNN được xử lý như phần discovery/toy operation, không đưa vào model chính vì không có cơ sở spatial/sequential trên raw representation.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation metrics và chọn ứng viên tốt nhất bằng validation F1.


Output cho thấy điều gì?

Bảng xếp hạng theo validation F1 vì dataset mất cân bằng và chỉ accuracy sẽ thiên về class 0. Trade-off cần đọc cùng Precision/Recall: recall cao hơn thường giảm false negatives nhưng có thể tăng false positives.


# 19. Representation Inspection

Cell này làm gì?

Inspect representation bên trong: activation của hidden layer cho model tốt nhất và attention weights nếu M4 được train.


Output cho thấy điều gì?

Hidden activations cho biết tabular vector được model biến đổi thành representation dày hơn trước output. Nếu có attention summary, đó là inspection phụ trợ về trọng số tương đối của feature trong M4, không phải bằng chứng nhân quả y khoa.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi trên validation set: false positives, false negatives và high-confidence errors của model được chọn theo validation.


Output cho thấy điều gì?

False negatives là nhóm cần chú ý trong bài toán diabetes vì model bỏ sót class 1 trên validation. High-confidence errors giúp nhận diện các vùng dữ liệu model tự tin nhưng sai; các nhận xét này chỉ dùng cho phân tích mô hình, không phải khuyến nghị lâm sàng.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation, evaluate test set đúng một lần, rồi lưu final model và metrics.


Output cho thấy điều gì?

Đây là kết quả test cuối cùng của model đã chọn bằng validation. `M3 Residual MLP` dùng threshold 0.63; test Accuracy 0.7844, Precision 0.3670, Recall 0.6084, F1 0.4578, ROC-AUC 0.8089.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc artifacts A03 nếu có và so sánh dataset/workflow/metrics với A04. Không kết luận A04 tốt hơn chỉ vì score khác, vì dataset đã đổi.


Output cho thấy điều gì?

A03 và A04 cùng bài toán nhưng khác dataset: A03 cân bằng hơn, A04 thực tế hơn và mất cân bằng hơn. Vì vậy metric không chứng minh deep learning intrinsically better/worse; nó chỉ mô tả workflow và kết quả trên dữ liệu A04 đã chọn.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


Output cho thấy điều gì?

Bảng cuối ghi lại dataset, model được chọn, threshold và test metrics thật. Artifact listing xác nhận notebook đã lưu model, metrics, figures và metadata cho phase Diabetes A04; recorded training/evaluation time khoảng 306.08 giây.


In [ ]:
import pandas as pd
from IPython.display import display, Image
_folder = _repo / 'A04/artifacts/pipeline04/diabetes/metrics'
for _file in sorted(_folder.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))


In [ ]:
display(Image(filename=str(_repo / 'A04/artifacts/pipeline04/diabetes/figures/keras_basic_mlp_training_curves.png')))
